In [ ]:
import json
import os
import glob
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Make the project root importable so `from src.bots import ...` works
sys.path.insert(0, str(Path.cwd().parent))

## Load data files & Mouse trajectory preview

In [ ]:
from src.plotting import plot_trajectory
from src.data import find_red_eclipse_files, load_red_eclipse_mouse

game_files = find_red_eclipse_files()

for file_path in game_files[:5]:
    meta, mouse = load_red_eclipse_mouse(file_path)
    if mouse.empty:
        continue
    plot_trajectory(mouse, title=f"userId={meta['userId']}, gameId={meta['gameId']}")
    print(f"\nuserId={meta['userId']}, gameId={meta['gameId']}, events={len(mouse)}")


## Feature extraction


In [ ]:
from src.features import extract_features
from src.data import load_red_eclipse_mouse

game_rows = []
for file_path in game_files:
    meta, mouse = load_red_eclipse_mouse(file_path)
    features = extract_features(mouse)

    if features is None:
        continue

    features.update({
        **meta,
        "is_bot": 0,
        "bot_type": "human",
    })
    game_rows.append(features)

games_df = pd.DataFrame(game_rows)
games_df.to_csv("../data/red_eclipse_features.csv", index=False)
print(games_df.head())


## Player identification


In [ ]:
from src.features import feature_cols

MIN_GAMES = 8
games_df_37 = games_df.groupby("userId").filter(lambda g: len(g) >= MIN_GAMES)

# games_df = all 45 players, games_df_37 = more than 8 games
select_model = games_df_37

input_data = select_model[feature_cols]
output_data = select_model["userId"]

input_train, input_test, output_train, output_test = train_test_split(
    input_data, output_data, test_size=0.2, random_state=42, stratify=output_data
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(input_train, output_train)

output_pred = model.predict(input_test)
accuracy = accuracy_score(output_test, output_pred)

print(f"Accuracy: {accuracy:.2%}")
print(f"Random baseline: {1 / output_data.nunique():.2%} ({output_data.nunique()} players, {len(select_model)} games)")
print()
print(classification_report(output_test, output_pred))


## Block-bootstrap synthetic bots


In [ ]:
from src.bots import build_segments, stitch_bot_game
from src.features import extract_features
from src.config import RNG_SEED
from src.data import load_red_eclipse_mouse

N_BOT_GAMES = len(games_df)  # one bot game per human game

rng = np.random.default_rng(RNG_SEED)

# 1) Build segment pool from human games
segment_pool = []
for file_path in game_files:
    meta, mouse = load_red_eclipse_mouse(file_path)
    segment_pool.extend(build_segments(mouse))

print(f"Segment pool: {len(segment_pool)} segments from {len(game_files)} games")

# 2) Generate synthetic bot games
N_PREVIEW = 5              # save first N trajectories for render graph
bot_rows = []
sample_bot_trajectories = []
for i in range(N_BOT_GAMES):
    bot_mouse = stitch_bot_game(segment_pool, rng=rng)
    if len(sample_bot_trajectories) < N_PREVIEW:
        sample_bot_trajectories.append(bot_mouse.copy())
    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({
        "userId": -1,
        "gameId": f"bot_{i}",
        "source_file": f"synthetic_bot_{i}",
        "is_bot": 1,
        "bot_type": "stitch",
    })
    bot_rows.append(feats)

bots_stitch_df = pd.DataFrame(bot_rows)
print(f"Generated {len(bots_stitch_df)} stitch bot games")
print(bots_stitch_df.head())


## Block-bootstrap synthetic bots trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_bot_trajectories):
    plot_trajectory(df, title=f"bot_{i}")
    print(f"\nbot_{i}, events={len(df)}")


## Smooth bot generation


In [ ]:
from src.bots import estimate_smooth_params, generate_smooth_bot_game
from src.features import extract_features

median_events = int(games_df["n_events"].median())
N_SMOOTH_BOTS = len(games_df)

re_smooth_params = estimate_smooth_params(games_df)
print(f"RE smooth params: {re_smooth_params}")

smooth_rows = []
sample_smooth_trajectories = []
for i in range(N_SMOOTH_BOTS):
    bot_mouse = generate_smooth_bot_game(n_events=median_events, seed=RNG_SEED + i, **re_smooth_params)
    if len(sample_smooth_trajectories) < N_PREVIEW:
        sample_smooth_trajectories.append(bot_mouse.copy())

    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({
        "userId": -2,
        "gameId": f"smooth_{i}",
        "source_file": f"synthetic_smooth_{i}",
        "is_bot": 1,
        "bot_type": "smooth",
    })
    smooth_rows.append(feats)

bots_smooth_df = pd.DataFrame(smooth_rows)
print(f"Generated {len(bots_smooth_df)} smooth bot games (n_events={median_events})")
print(bots_smooth_df.head())


## Smooth bot trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_smooth_trajectories):
    plot_trajectory(df, title=f"smooth_{i}")
    print(f"\nsmooth_{i}, events={len(df)}")

## Human vs Bot classification


In [ ]:
from src.evaluation import train_human_vs_bot
from src.features import feature_cols

model_stitch, acc_stitch = train_human_vs_bot(
    games_df, bots_stitch_df, feature_cols, name="stitch"
)
model_smooth, acc_smooth = train_human_vs_bot(
    games_df, bots_smooth_df, feature_cols, name="smooth"
)

print("Bot detection difficulty (higher = easier to catch)")
print(f"  stitch: {acc_stitch:.2%}")
print(f"  smooth: {acc_smooth:.2%}")


## Load LoL dataset

In [ ]:
from src.data import parse_lol_keylogger, find_lol_keylogger_files

RE_TRAIN_N = None
LOL_FILE_N = None
LOL_WINDOW_MIN = 3
LOL_PARSE_MAX_EVENTS = None
N_LOL_BOTS = None
SKIP_FULL_LOL_TRAJECTORY = False

lol_keylogger_files = find_lol_keylogger_files()

sample_lol = parse_lol_keylogger(
    lol_keylogger_files[0],
    max_events=LOL_PARSE_MAX_EVENTS,
    max_minutes=LOL_WINDOW_MIN,
)
print(f"File: {lol_keylogger_files[0].name}")
print(f"Events: {len(sample_lol)}, duration: {sample_lol['time'].iloc[-1] / 60000:.1f} min")
print(sample_lol.head())


## Load LoL sessions & extract features

In [ ]:
from src.features import extract_features
from src.data import parse_lol_keylogger, find_lol_keylogger_files

lol_keylogger_files = find_lol_keylogger_files()

files_to_load = lol_keylogger_files[:LOL_FILE_N] if LOL_FILE_N else lol_keylogger_files
print(f"Loading {len(files_to_load)} LoL files")

lol_rows = []
for file_path in files_to_load:
    mouse = parse_lol_keylogger(
        file_path,
        max_events=LOL_PARSE_MAX_EVENTS,
        max_minutes=LOL_WINDOW_MIN,
    )
    if mouse is None:
        continue
    mouse = mouse[mouse["time"] <= LOL_WINDOW_MIN * 60 * 1000]
    feats = extract_features(mouse)
    if feats is None:
        continue

    session_date = file_path.parent.name
    participant = file_path.stem.split("-")[1]
    feats.update({
        "userId": f"lol_{participant}",
        "gameId": f"{session_date}_p{participant}",
        "source_file": file_path.name,
        "session_date": session_date,
        "is_bot": 0,
        "bot_type": "human",
    })
    lol_rows.append(feats)

lol_games_df = pd.DataFrame(lol_rows)
print(f"Loaded {len(lol_games_df)} LoL human sessions (first {LOL_WINDOW_MIN} min each)")
print(lol_games_df[["n_events", "total_movement", "avg_speed", "idle_ratio"]].describe())
print()
print(f"Red Eclipse — median n_events: {games_df['n_events'].median():.0f}")
print(f"LoL — median n_events: {lol_games_df['n_events'].median():.0f}")


## LoL trajectory preview

In [ ]:
from src.data import parse_lol_keylogger, find_lol_keylogger_files

lol_keylogger_files = find_lol_keylogger_files()

PREVIEW_MINUTES = LOL_WINDOW_MIN

preview_lol = parse_lol_keylogger(
    lol_keylogger_files[0],
    max_events=LOL_PARSE_MAX_EVENTS,
    max_minutes=PREVIEW_MINUTES,
)
preview_lol = preview_lol[preview_lol["time"] <= PREVIEW_MINUTES * 60 * 1000].copy()
preview_lol["trajectory_x"] = preview_lol["dx"].cumsum()
preview_lol["trajectory_y"] = preview_lol["dy"].cumsum()

print(f"Zoom: first {PREVIEW_MINUTES} min, {len(preview_lol)} events")

if SKIP_FULL_LOL_TRAJECTORY:
    plt.figure(figsize=(7, 5))
    plt.plot(preview_lol["trajectory_x"], preview_lol["trajectory_y"], linewidth=0.5, color="steelblue")
    plt.title(f"LoL — first {PREVIEW_MINUTES} min")
    plt.gca().invert_yaxis()
    plt.axis("equal")
    plt.show()
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(preview_lol["trajectory_x"], preview_lol["trajectory_y"], linewidth=0.5, color="steelblue")
    axes[0].set_title(f"LoL — first {PREVIEW_MINUTES} min")
    axes[0].invert_yaxis()
    axes[0].set_aspect("equal")
    full_lol = parse_lol_keylogger(lol_keylogger_files[0])
    full_lol["trajectory_x"] = full_lol["dx"].cumsum()
    full_lol["trajectory_y"] = full_lol["dy"].cumsum()
    axes[1].plot(full_lol["trajectory_x"], full_lol["trajectory_y"], linewidth=0.1, color="steelblue", alpha=0.3)
    axes[1].set_title(f"LoL — full session ({len(full_lol)/1e6:.2f}M events)")
    axes[1].invert_yaxis()
    axes[1].set_aspect("equal")
    plt.tight_layout()
    plt.show()


## Cross-game transfer (RE train → LoL test)

In [ ]:
from src.features import cross_game_feature_cols
from src.evaluation import train_human_vs_bot

re_human = games_df.head(RE_TRAIN_N) if RE_TRAIN_N else games_df
re_stitch = bots_stitch_df.head(RE_TRAIN_N) if RE_TRAIN_N else bots_stitch_df
re_smooth = bots_smooth_df.head(RE_TRAIN_N) if RE_TRAIN_N else bots_smooth_df

print("=== RE model trained on STITCH bots ===")
model_cross_stitch, _ = train_human_vs_bot(re_human, re_stitch, cross_game_feature_cols, name="stitch")
print()
print("=== RE model trained on SMOOTH bots ===")
model_cross_smooth, _ = train_human_vs_bot(re_human, re_smooth, cross_game_feature_cols, name="smooth")


## LoL data human identifier

In [ ]:
# test will LoL humans identified as bot
for name, model in [("stitch", model_cross_stitch), ("smooth", model_cross_smooth)]:
    pred = model.predict(lol_games_df[cross_game_feature_cols])
    print(f"[{name} model] LoL human false-positive rate: {pred.mean():.2%} ({pred.sum()}/{len(pred)} identified as bot)")

## LoL stitch bot generation

In [ ]:
from src.bots import build_segments, stitch_bot_game
from src.data import parse_lol_keylogger

lol_segment_pool = []
for file_path in files_to_load:
    mouse = parse_lol_keylogger(
        file_path,
        max_events=LOL_PARSE_MAX_EVENTS,
        max_minutes=LOL_WINDOW_MIN,
    )
    if mouse is None:
        continue
    mouse = mouse[mouse["time"] <= LOL_WINDOW_MIN * 60 * 1000]
    lol_segment_pool.extend(build_segments(mouse))

n_lol_bots = N_LOL_BOTS if N_LOL_BOTS else len(lol_games_df)
target_ms = LOL_WINDOW_MIN * 60 * 1000
lol_stitch_rows = []
sample_lol_stitch_trajectories = []
lol_bot_rng = np.random.default_rng(RNG_SEED + 1)

for i in range(n_lol_bots):
    bot_mouse = stitch_bot_game(lol_segment_pool, target_duration_ms=target_ms, rng=lol_bot_rng)
    if len(sample_lol_stitch_trajectories) < N_PREVIEW:
        sample_lol_stitch_trajectories.append(bot_mouse.copy())
    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({"userId": -1, "gameId": f"lol_stitch_{i}", "is_bot": 1, "bot_type": "stitch"})
    lol_stitch_rows.append(feats)

lol_bots_stitch_df = pd.DataFrame(lol_stitch_rows)
print(f"LoL stitch bots: {len(lol_bots_stitch_df)} (target {target_ms/1000:.0f}s each)")

## LoL stitch bot trajectory preview

In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_lol_stitch_trajectories):
    print(f"lol_stitch_{i}, events={len(df)}")
    plot_trajectory(df, title=f"lol_stitch_{i}")

## LoL smooth bot generation

In [ ]:
from src.bots import estimate_smooth_params, generate_smooth_bot_game

lol_median_events = int(lol_games_df["n_events"].median())
n_lol_smooth = N_LOL_BOTS if N_LOL_BOTS else len(lol_games_df)

lol_smooth_params = estimate_smooth_params(lol_games_df)
print(f"LoL smooth params: {lol_smooth_params}")
print(f"(RE smooth params for comparison: {re_smooth_params})")

lol_smooth_rows = []
sample_lol_smooth_trajectories = []
for i in range(n_lol_smooth):
    bot_mouse = generate_smooth_bot_game(n_events=lol_median_events, seed=RNG_SEED + 100 + i, **lol_smooth_params)
    if len(sample_lol_smooth_trajectories) < N_PREVIEW:
        sample_lol_smooth_trajectories.append(bot_mouse.copy())
    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({"userId": -2, "gameId": f"lol_smooth_{i}", "is_bot": 1, "bot_type": "smooth"})
    lol_smooth_rows.append(feats)

lol_bots_smooth_df = pd.DataFrame(lol_smooth_rows)
print(f"LoL smooth bots: {len(lol_bots_smooth_df)} (n_events={lol_median_events})")

## LoL smooth bot trajectory preview

In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_lol_smooth_trajectories):
    plot_trajectory(df, title=f"lol_smooth_{i}")
    print(f"lol_smooth_{i}, events={len(df)}")


## Cross-game detection per bot tier

In [ ]:
tiers = [
    ("stitch", model_cross_stitch, lol_bots_stitch_df),
    ("smooth", model_cross_smooth, lol_bots_smooth_df),
]

print("Zero-shot bot detection (RE-trained -> LoL, no retrain):")
for name, model, bot_df in tiers:
    pred = model.predict(bot_df[cross_game_feature_cols])
    print(f"  {name:7s}: {pred.mean():.2%} caught ({pred.sum()}/{len(pred)})")

print()
print("Reference — same bot tier IN-DOMAIN on Red Eclipse:")
print("  stitch : ~85%")
print("  smooth : ~100%")

## LoL in-domain sanity check

In [ ]:
from src.evaluation import train_human_vs_bot
from src.features import feature_cols

_, lol_acc_stitch = train_human_vs_bot(
    lol_games_df, lol_bots_stitch_df, feature_cols, name="LoL stitch"
)
print()
_, lol_acc_smooth = train_human_vs_bot(
    lol_games_df, lol_bots_smooth_df, feature_cols, name="LoL smooth"
)

print()
print("LoL in-domain bot detection (train+test on LoL):")
print(f"  stitch: {lol_acc_stitch:.2%}")
print(f"  smooth: {lol_acc_smooth:.2%}")
print()
print("Compare with zero-shot (RE-trained -> LoL): both ~0%")
print("-> high in-domain here means the failure is cross-game transfer, not the data.")
